In [1]:
# Step 1: Import required libraries
import pandas as pd
import string

# Load your datasets
places = pd.read_csv("places.csv")
reviews = pd.read_csv("reviews.csv")

In [2]:
places.head()

,place_id,name,lat,lng,address,category,types,rating,user_ratings_total,phone,website
0,ChIJF2_wv4lHDW0R47WKR5aSdmE,Onslow,-36.847685,174.769957,"9 Princes Street, Auckland Central, Auckland 1...",restaurant,"establishment,food,point_of_interest,restaurant",4.8,1331,+64 9 930 9123,http://www.onslow.nz/
1,ChIJKZLQLLRHDW0R_F9vtz1cjiQ,alma,-36.844086,174.769165,"130 Quay Street, Auckland Central, Auckland 10...",restaurant,"establishment,food,point_of_interest,restaurant",4.7,413,+64 9 242 1570,http://www.alma.nz/
2,ChIJS_i53flHDW0R6H8fEWgxVfk,Soul Bar & Bistro,-36.843406,174.763017,"Corner of Lower Hobson Street, Customs Street ...",restaurant,"bar,establishment,food,point_of_interest,resta...",4.5,2338,+64 9 356 7249,http://www.soulbar.co.nz/
3,ChIJn3Evd2xHDW0RWNG-B0TrPxg,Ahi.,-36.843562,174.766454,"Commercial Bay Level 2/7 Queen Street, Aucklan...",restaurant,"establishment,food,point_of_interest,restaurant",4.6,1106,+64 22 524 4255,http://ahi.restaurant/
4,ChIJzV2foiBGDW0ROPBzbgZDJuc,One Tree Grill,-36.899825,174.772730,"9 Pah Road, Epsom, Auckland 1023, New Zealand",restaurant,"establishment,food,point_of_interest,restaurant",4.8,1604,+64 9 909 7215,https://www.onetreegrill.co.nz/


In [3]:
reviews.head()

,review_id,place_id,text,rating,lang,publish_time_utc,author_name,review_photo_url,review_time
0,1cc99ebc14f41739906d0f99,ChIJ1QL4cyBdDW0RcU21rEmrs2A,Had a 2 day intensive team building camp. Grea...,5,en,1708136374,Andy Parr,NaN,1708136374
1,03e683633d3b2d748e132486,ChIJ1QL4cyBdDW0RcU21rEmrs2A,We are still here just absolutely lovely,5,en,1704774509,Katherine Taniwha,NaN,1704774509
2,1f912911415e322b69ac5b3c,ChIJL8iQ5ElfDW0RGehL9xBV-lY,We’ll it ready depends what you are looking fo...,5,en,1647137779,Hamada Eleleimy,NaN,1647137779
3,c098297abf137bf5cb1ea2b6,ChIJL8iQ5ElfDW0RGehL9xBV-lY,Love this place and we keep coming back 3 time...,5,en,1742178610,Toakase Tauheluhelu,NaN,1742178610
4,b6ac7c1efc5a91704cb0796a,ChIJL8iQ5ElfDW0RGehL9xBV-lY,"Beautiful place to stay, lovely view, short wa...",5,en,1719103180,age 50,NaN,1719103180


## Data Preprocessing

Group reviews

In [4]:
# Group reviews to summarize per place_id
reviews_grouped = (
    reviews
    .groupby("place_id", as_index=False)
    .agg({
        "review_id": "count",            # how many reviews per shop
        "text": lambda x: list(x)        # optional: keep list of review texts
    })
    .rename(columns={"review_id": "review_count"})
)
reviews_grouped.head()


,place_id,review_count,text
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,6,[We had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,8,[Service was great! Guy at the cashier and ser...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi..."


In [5]:
# Merge grouped reviews with places, now also including 'category'
shops_r = reviews_grouped.merge(
    places[["place_id", "name", "address", "lat", "lng", "user_ratings_total", "category"]],
    on="place_id",
    how="inner",
    validate="1:1"
)
shops_r.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park
1,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87,tourist attraction
2,ChIJ-1xcOwA7DW0RbE_4woT8gl8,1,[I had an awesome time at the Albany Marathon....,Albany Marathon,"North Shore, Albany, Auckland 0632, New Zealand",-36.726957,174.708765,1,park
3,ChIJ-4Ali4xHDW0RVUTQNlJD1ms,5,[Attending the recent Furry Convention was an ...,Cooper Reserve,"28 Seddon Street, Grey Lynn, Auckland 1021, Ne...",-36.867424,174.744799,9,park
4,ChIJ-4wZDc9BDW0RNpsXE2YTFs8,4,[I pass Shona reserve on my favourite twin str...,Shona Reserve,"16 Chardon Place, Henderson, Auckland 0612, Ne...",-36.884224,174.619895,10,park


### Quick overview of the data

In [6]:
print("Places dataset-")
print(f"Rows: {places.shape[0]}, Columns: {places.shape[1]}")

print("Reviews dataset-")
print(f"Rows: {reviews.shape[0]}, Columns: {reviews.shape[1]}")

print("Grouped reviews dataset-")
print(f"Rows: {reviews_grouped.shape[0]}, Columns: {reviews_grouped.shape[1]}")

print("Merged dataset-")
print(f"Rows: {shops_r.shape[0]}, Columns: {shops_r.shape[1]}")

Places dataset-
Rows: 1528, Columns: 11
Reviews dataset-
Rows: 27104, Columns: 9
Grouped reviews dataset-
Rows: 5373, Columns: 3
Merged dataset-
Rows: 773, Columns: 9


**Check for missingness**

In [7]:
# Count missing values in each column
shops_r.isna().sum()


place_id              0
review_count          0
text                  0
name                  0
address               0
lat                   0
lng                   0
user_ratings_total    0
category              0
dtype: int64

**Check for duplicates**

In [8]:
# Verify one unique row per place
duplicate_places = shops_r["place_id"].duplicated().sum()
print(f"Duplicate place_ids: {duplicate_places}")


Duplicate place_ids: 0


In [9]:
#print("Max reviews seen:", shops_r["review_count"].max())
#assert shops_r["review_count"].max() <= 5, "Unexpected: a place has >5 reviews"


In [10]:
# Find places with >5 reviews
shops_r[shops_r["review_count"] > 5][["place_id", "name", "review_count"]]


,place_id,name,review_count
22,ChIJ0Z9eaQJKDW0RDo_BLkyDYuU,La Vista Cafe & Restaurant,9
50,ChIJ33d6GUZJDW0ROhg2g73UHKU,Casablanca Sylvia Park,9
64,ChIJ4VwbgIVHDW0RbrSpXbsyD5s,Jervois Steak House,7
111,ChIJ8TXqG7FGDW0R4dFrYZg_bF0,Malaysian Noodles & Rice House,6
123,ChIJ99NgkUhJDW0RRXha4aifKFo,Baci Eatery,7
146,ChIJB3LhgY1HDW0RI5-7-ck-G_w,Queenies Caffé & Vino,7
156,ChIJC-NTSulLDW0Rxs_PbYedU_M,Porterhouse Grill,6
172,ChIJDRTsY5E7DW0RYIideME3c7g,Lone Star Albany,8
204,ChIJFdEVBx5JDW0RGDRj2nbOLhk,Ellerslie International,6
227,ChIJHRUkM-RJDW0RQ-e4KNalPzI,Kohi Beach Eatery & Store,6


In [11]:
# Look at the actual review texts for one
pid = shops_r.loc[shops_r["review_count"] > 5, "place_id"].iloc[0]
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == pid, "text"].values[0], start=1):
    print(f"{idx}. {review}\n")


1. Beautiful spot right by the beach in St Heliers. Love coming here for brunch, but dinner was a whole new level – fresh tiger prawns, perfectly cooked eye fillet and market fish, and the most amazing light tiramisu with strawberry sauce. Great food, romantic atmosphere, and stunning sea views!

2. Our lunch at La Vista Cafe was OMG delicious. The reviews are very complimentary about the cafe and I 💯 % agree. We had Tapas to include smoked fish cakes, patatas bravas, lamb and chorizio dumplings, albondigas and pizza focaccia. All fabulous and super generous portions (we actually brought some home so we could enjoy it again later). Service was excellent and a lovely feel to the cafe. I actually like all the nice lighting arrangements!

3. 100% recommend this wonderful café! We had an amazing time there with my wife, drinking some NZ's finest wines with super tasty fresh market fish! I also had a perfect steak! Dessert tiramisu was a top notch as well - the best we tasted in Auckland so

### Text Cleaning and Preprocessing

In [12]:
import re

def clean_review_text(text):
    """Normalize review text for NLP."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()           # normalize spaces
    return text

# Create a new column with cleaned reviews for each shop
shops_r["clean_texts"] = shops_r["text"].apply(lambda reviews: [clean_review_text(r) for r in reviews])
shops_r.head()


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...
1,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87,tourist attraction,[great place for families and dog walkers benc...
2,ChIJ-1xcOwA7DW0RbE_4woT8gl8,1,[I had an awesome time at the Albany Marathon....,Albany Marathon,"North Shore, Albany, Auckland 0632, New Zealand",-36.726957,174.708765,1,park,[i had an awesome time at the albany marathon ...
3,ChIJ-4Ali4xHDW0RVUTQNlJD1ms,5,[Attending the recent Furry Convention was an ...,Cooper Reserve,"28 Seddon Street, Grey Lynn, Auckland 1021, Ne...",-36.867424,174.744799,9,park,[attending the recent furry convention was an ...
4,ChIJ-4wZDc9BDW0RNpsXE2YTFs8,4,[I pass Shona reserve on my favourite twin str...,Shona Reserve,"16 Chardon Place, Henderson, Auckland 0612, Ne...",-36.884224,174.619895,10,park,[i pass shona reserve on my favourite twin str...


In [13]:
shops_r.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...
1,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87,tourist attraction,[great place for families and dog walkers benc...
2,ChIJ-1xcOwA7DW0RbE_4woT8gl8,1,[I had an awesome time at the Albany Marathon....,Albany Marathon,"North Shore, Albany, Auckland 0632, New Zealand",-36.726957,174.708765,1,park,[i had an awesome time at the albany marathon ...
3,ChIJ-4Ali4xHDW0RVUTQNlJD1ms,5,[Attending the recent Furry Convention was an ...,Cooper Reserve,"28 Seddon Street, Grey Lynn, Auckland 1021, Ne...",-36.867424,174.744799,9,park,[attending the recent furry convention was an ...
4,ChIJ-4wZDc9BDW0RNpsXE2YTFs8,4,[I pass Shona reserve on my favourite twin str...,Shona Reserve,"16 Chardon Place, Henderson, Auckland 0612, Ne...",-36.884224,174.619895,10,park,[i pass shona reserve on my favourite twin str...


### Tokenisation

In [14]:
import re

# A basic stopword list (can be expanded later)
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below
between both but by can did do does doing down during each few for from further had has have having
he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not of off on once only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there these they this those through to too
under until up very was we were what when where which while who whom why will with you your yours
yourself yourselves
""".split())

def tokenize_and_remove_stopwords(text):
    """Split text into tokens and remove common stopwords."""
    tokens = re.findall(r"[a-z']+", text)  # words only
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens


Clean empty reviews

In [15]:
# Remove shops where any review text in the list is empty or NaN
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda reviews: [r for r in reviews if r.strip() not in ("", "nan")]
)

# Drop rows where the resulting list is empty (no valid reviews left)
shops_r = shops_r[shops_r["clean_texts"].apply(len) > 0].copy()


In [16]:
# Apply to each review in every shop
shops_r["tokens"] = shops_r["clean_texts"].apply(
    lambda reviews: [tokenize_and_remove_stopwords(r) for r in reviews]
)

shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...,"[[favourite, place, take, little, one, quick, ..."


In [17]:
# Look at tokens for a random shop
import random
sample_tokens = random.choice(shops_r["tokens"].values)
for i, review_tokens in enumerate(sample_tokens, start=1):
    print(f"Review {i}: {review_tokens}")


Review 1: ['nice', 'memorial', 'local', 'rnzaf', 'airmen', 'killed', 'action', 'ww', 'nice', 'tribute', 'nice', 'location', 'next', 'historic', 'gun', 'emplacements']
Review 2: ['cool', 'park', 'get', 'quite', 'wet', 'muddy', 'rains']


Drop single character tokens

In [18]:
import re

def clean_tokens(tokens):
    out = []
    for t in tokens:
        if t == "s":               # drop possessive leftovers
            continue
        if len(t) < 2:             # drop 1-char tokens
            continue
        if re.fullmatch(r"\d+", t):# drop pure numbers
            continue
        out.append(t)
    return out

# apply to each review’s token list
shops_r["tokens_clean"] = shops_r["tokens"].apply(lambda reviews: [clean_tokens(toks) for toks in reviews])
shops_r = shops_r.drop('tokens', axis=1)
shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...,"[[favourite, place, take, little, one, quick, ..."


**Categorizing the places and reviews**

In [19]:
# Create four different DataFrames by category
rest_df= shops_r[shops_r["category"]== "restaurant"]
park_df= shops_r[shops_r["category"]== "park"]
mall_df= shops_r[shops_r["category"]== "shopping mall"]
tour_df= shops_r[shops_r["category"]== "tourist attraction"]

test_df=rest_df.copy()


In [20]:
# Show counts for each category
print("Restaurants:", len(rest_df))
print("Parks:", len(park_df))
print("Shopping Malls:", len(mall_df))
print("Tourist Attractions:", len(tour_df))

Restaurants: 38
Parks: 452
Shopping Malls: 48
Tourist Attractions: 235


In [21]:
# Restaurants
rest_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,5,[Fair prices and great service! Food was tasty...,The Corner Bar and Eatery,"163 Queens Road, Panmure, Auckland 1072, New Z...",-36.901067,174.856835,144,restaurant,[fair prices and great service food was tasty ...,"[[fair, prices, great, service, food, tasty, g..."
22,ChIJ0Z9eaQJKDW0RDo_BLkyDYuU,9,[Beautiful spot right by the beach in St Helie...,La Vista Cafe & Restaurant,"417 Tamaki Drive, St Heliers, Auckland 1071, N...",-36.850229,174.857834,958,restaurant,[beautiful spot right by the beach in st helie...,"[[beautiful, spot, right, beach, st, heliers, ..."
40,ChIJ26BROdxJDW0RROHKm1Nl8yE,5,"[Four of us came for dinner, and the food was ...",Mission Bay Cafe,"85 Tamaki Drive, Mission Bay, Auckland 1071, N...",-36.848396,174.832260,1103,restaurant,[four of us came for dinner and the food was a...,"[[four, us, came, dinner, food, absolutely, ex..."
50,ChIJ33d6GUZJDW0ROhg2g73UHKU,9,[Hey everyone! I just wanted to share my exper...,Casablanca Sylvia Park,"Sylvia Park Shopping Centre 66 Dining Lane, Mo...",-36.914845,174.840779,1849,restaurant,[hey everyone i just wanted to share my experi...,"[[hey, everyone, wanted, share, experience, am..."
64,ChIJ4VwbgIVHDW0RbrSpXbsyD5s,7,[My partner and I visited Jervois Steakhouse o...,Jervois Steak House,"70 Jervois Road, Ponsonby, Auckland 1011, New ...",-36.846168,174.741887,1515,restaurant,[my partner and i visited jervois steakhouse o...,"[[partner, visited, jervois, steakhouse, busy,..."


In [22]:
# Parks
park_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...,"[[favourite, place, take, little, one, quick, ..."
2,ChIJ-1xcOwA7DW0RbE_4woT8gl8,1,[I had an awesome time at the Albany Marathon....,Albany Marathon,"North Shore, Albany, Auckland 0632, New Zealand",-36.726957,174.708765,1,park,[i had an awesome time at the albany marathon ...,"[[awesome, time, albany, marathon, early, morn..."
3,ChIJ-4Ali4xHDW0RVUTQNlJD1ms,5,[Attending the recent Furry Convention was an ...,Cooper Reserve,"28 Seddon Street, Grey Lynn, Auckland 1021, Ne...",-36.867424,174.744799,9,park,[attending the recent furry convention was an ...,"[[attending, recent, furry, convention, experi..."
4,ChIJ-4wZDc9BDW0RNpsXE2YTFs8,4,[I pass Shona reserve on my favourite twin str...,Shona Reserve,"16 Chardon Place, Henderson, Auckland 0612, Ne...",-36.884224,174.619895,10,park,[i pass shona reserve on my favourite twin str...,"[[pass, shona, reserve, favourite, twin, strea..."
5,ChIJ-5DkQwA5DW0RI8Jo-PTSQQM,1,[Okay. It is wild and overgrown. And the track...,Duud Reserve,"24 Marlborough Avenue, Glenfield, Auckland 062...",-36.784391,174.724200,1,park,[okay it is wild and overgrown and the track i...,"[[okay, wild, overgrown, track, also, overgrow..."


In [23]:
# Malls
mall_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
18,ChIJ0-4gpPJGDW0RUY3nD0-Pn1k,5,[This is a very nice center. There are not too...,Tulja Centre,"190 Stoddard Road, Wesley, Auckland 1041, New ...",-36.900680,174.721294,44,shopping mall,[this is a very nice center there are not too ...,"[[nice, center, many, businesses, couple, time..."
19,ChIJ04jFZFS1cm0RjCaU6uIWLeo,5,[Dropped by this lovely & relatively new small...,Pohutukawa Coast Shopping Centre,"129 Beachlands Road, Beachlands 2018, New Zealand",-36.889547,175.010820,10,shopping mall,[dropped by this lovely relatively new small s...,"[[dropped, lovely, relatively, new, small, sho..."
45,ChIJ2YxMszFLDW0RXc9zIlvgsVI,5,"[Avoid the Burger Fuel, worst service ever. 30...",Kentigern Plaza,"102 Pakuranga Road, Pakuranga, Auckland 2010, ...",-36.910881,174.871609,54,shopping mall,[avoid the burger fuel worst service ever 30mi...,"[[avoid, burger, fuel, worst, service, ever, m..."
62,ChIJ42jp8DA5DW0REj0s01g7RMk,5,[Nanda has been doing my brows for almost a ye...,Vish Beauty Bar,"Glenfield Mall Glenfield Rd and, Downing Stree...",-36.783504,174.721428,71,shopping mall,[nanda has been doing my brows for almost a ye...,"[[nanda, brows, almost, year, now, always, don..."
71,ChIJ554ImrNBDW0R910mtUnpvZo,3,"[Great place for shopping, Great shops under o...",Shopping Plaza Arcade,"357 Great North Road, Henderson, Auckland 0612...",-36.879660,174.633223,9,shopping mall,"[great place for shopping, great shops under o...","[[great, place, shopping], [great, shops, one,..."


In [24]:
# Tourist Attractions
tour_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
1,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87,tourist attraction,[great place for families and dog walkers benc...,"[[great, place, families, dog, walkers, benche..."
10,ChIJ-XCDIlBBDW0Rybt9IcvRgog,5,[Always an interesting place to visit as I’m i...,Te Toi Uku - Crown Lynn & Clayworks Museum,"8 Ambrico Place, New Lynn, Auckland 0600, New ...",-36.911109,174.680283,72,tourist attraction,[always an interesting place to visit as i m i...,"[[always, interesting, place, visit, intereste..."
12,ChIJ-ZvaR5FJDW0RImTC-QQrWYI,5,[Perfect place for an hour in nature amongst r...,St Johns Bush Walk,"133 Gowing Drive, Meadowbank, Auckland 1072, N...",-36.870544,174.841576,99,tourist attraction,[perfect place for an hour in nature amongst r...,"[[perfect, place, hour, nature, amongst, resid..."
13,ChIJ-_KMDFo4DW0ROFkm2fVfLBk,1,[Amazing sculptures and artwork. Darryl has an...,FAGENCE ART - Sculpture Garden & Gallery,"18 Paruru Avenue, Northcote, Auckland 0627, Ne...",-36.806137,174.737953,5,tourist attraction,[amazing sculptures and artwork darryl has an ...,"[[amazing, sculptures, artwork, darryl, great,..."
20,ChIJ08wIYwBHDW0RgVW8RBW0jj8,2,[Sing the song and watch/hear the beautiful ar...,Waimahara,"Mayoral Drive, Auckland Central, Auckland 1010...",-36.854013,174.762298,5,tourist attraction,[sing the song and watch hear the beautiful ar...,"[[sing, song, watch, hear, beautiful, artwork,..."


### Keyword extraction

In [25]:
# Define keywords
KEYWORDS = {
    "restaurant": {"clean", "cozy", "spacious", "crowded", "busy", "quiet", "noisy", "friendly", "rude",
        "welcoming", "attentive", "professional", "efficient", "slow", "quick", "prompt", "helpful",
        "approachable", "hygienic", "dirty", "tidy", "spotless", "comfortable", "uncomfortable", "overpriced",
        "expensive", "cheap", "affordable", "reasonable", "value", "worthwhile", "overrated", "excellent",
        "great", "amazing", "lovely", "fantastic", "outstanding", "average", "decent", "disappointing",
        "terrible", "awful", "poor", "superb", "nice", "pleasant", "beautiful", "gorgeous", "atmospheric",
        "decorated", "modern", "traditional", "inviting", "stylish", "safe", "unsafe"},
    "park": {"green", "lush", "clean", "peaceful", "quiet", "serene", "tranquil", "calm", "relaxing",
        "spacious", "open", "crowded", "busy", "safe", "unsafe", "family-friendly", "child-friendly",
        "pet-friendly", "dog-friendly", "accessible", "inclusive", "welcoming", "tidy", "hygienic", "dirty",
        "polluted", "beautiful", "scenic", "picturesque", "refreshing", "natural", "breezy", "shady",
        "sunny", "greenery", "landscaped", "maintained", "unkept", "secure", "unsafe", "peaceful",
        "quiet", "relaxing", "recreational", "sporty", "vibrant", "energetic", "playful", "safe",
        "clean", "refreshing", "lovely"},
    "shopping mall": {"spacious", "crowded", "busy", "quiet", "safe", "secure", "unsafe", "modern", "stylish",
        "clean", "tidy", "dirty", "hygienic", "accessible", "inclusive", "welcoming", "organized", "chaotic",
        "bright", "well-lit", "dark", "confusing", "easy", "navigable", "sprawling", "compact", "big",
        "huge", "small", "cramped", "air-conditioned", "comfortable", "uncomfortable", "overpriced", "expensive",
        "affordable", "cheap", "reasonable", "family-friendly", "child-friendly", "crowded", "popular", "busy",
        "trendy", "upscale", "luxury", "basic", "ordinary", "modernized", "outdated", "stylish", "inviting",
        "safe", "clean", "friendly"},
    "tourist attraction": {"historic", "ancient", "modern", "beautiful", "gorgeous", "scenic", "picturesque", "breathtaking",
        "majestic", "grand", "iconic", "famous", "popular", "crowded", "busy", "peaceful", "quiet", "serene",
        "clean", "tidy", "dirty", "unsafe", "safe", "secure", "accessible", "welcoming", "inclusive", "touristy",
        "authentic", "cultural", "traditional", "vibrant", "colorful", "energetic", "spiritual", "sacred",
        "artistic", "creative", "inspiring", "memorable", "remarkable", "unique", "extraordinary", "ordinary",
        "overrated", "expensive", "affordable", "reasonable", "educational", "informative", "guided", "interactive",
        "family-friendly", "child-friendly", "adventurous", "photogenic"}
}

In [26]:
def add_keywords(df, category):
    vocab = KEYWORDS[category]
    df = df.copy()
    df["keywords"] = df["tokens_clean"].apply(
        lambda reviews: sorted({w for toks in reviews for w in toks if w in vocab})
    )
    return df


In [27]:
rest_df = add_keywords(rest_df, "restaurant")
park_df = add_keywords(park_df, "park")
mall_df = add_keywords(mall_df, "shopping mall")
tour_df = add_keywords(tour_df, "tourist attraction")

test_df = add_keywords(test_df, "restaurant")

In [28]:
test_df[["place_id","name","keywords"]].head()

,place_id,name,keywords
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,The Corner Bar and Eatery,"[amazing, clean, friendly, great, reasonable]"
22,ChIJ0Z9eaQJKDW0RDo_BLkyDYuU,La Vista Cafe & Restaurant,"[amazing, beautiful, cheap, comfortable, cozy,..."
40,ChIJ26BROdxJDW0RROHKm1Nl8yE,Mission Bay Cafe,"[attentive, average, clean, excellent, great, ..."
50,ChIJ33d6GUZJDW0ROhg2g73UHKU,Casablanca Sylvia Park,"[amazing, attentive, beautiful, cozy, excellen..."
64,ChIJ4VwbgIVHDW0RbrSpXbsyD5s,Jervois Steak House,"[amazing, busy, comfortable, excellent, fantas..."


## Sentiment analysis

In [29]:
!pip install vaderSentiment


In [30]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# initialize analyzer
analyzer = SentimentIntensityAnalyzer()


In [31]:
def score_keywords(keywords):
    if isinstance(keywords, str):
        keywords = [keywords]
    elif not isinstance(keywords, list):
        return []

    return [analyzer.polarity_scores(kw)["compound"] for kw in keywords if isinstance(kw, str)]


In [32]:
test_df["kw_scores"] = test_df["keywords"].apply(score_keywords)


In [33]:
# peek one row
test_df[["place_id","name","kw_scores"]].head(1)

,place_id,name,kw_scores
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,The Corner Bar and Eatery,"[0.5859, 0.4019, 0.4939, 0.6249, 0.0]"


In [34]:
test_df["keywords"].iloc[0]  # This should be a list of lists, not a long string

['amazing', 'clean', 'friendly', 'great', 'reasonable']

In [35]:
# Save to CSV
test_df.to_csv("test_df_with_keyword_scores.csv", index=False)


In [36]:
def aggregate_sentiment(kw_scores):
    scores = []
    for score in kw_scores:
        if isinstance(score, list):
            scores.extend(score)
        elif isinstance(score, (int, float)):
            scores.append(score)

    if not scores:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])

    avg = sum(scores) / len(scores)
    pos = sum(1 for s in scores if s > 0.05) / len(scores)
    neg = sum(1 for s in scores if s < -0.05) / len(scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([avg, pos, neg, label],
                     index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])


In [37]:
test_df[["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]] = (
    test_df["kw_scores"].apply(aggregate_sentiment)
)

# Repeat similarly for park_df, mall_df, tour_df if needed


In [38]:
test_df[["place_id","name","kw_scores","avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]].head(1)

,place_id,name,kw_scores,avg_sentiment,pct_positive,pct_negative,sentiment_label
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,The Corner Bar and Eatery,"[0.5859, 0.4019, 0.4939, 0.6249, 0.0]",0.42132,0.8,0.0,positive


#### SAMPLE TESTING

In [39]:
import random

random.seed(42)
# Randomly select 30 unique place_ids
sample_ids = random.sample(list(test_df["place_id"].dropna().unique()), 30)
sample_df = test_df[test_df["place_id"].isin(sample_ids)]

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)


#### TEXTBLOB

In [40]:
from textblob import TextBlob

In [41]:
def get_textblob_sentiment(texts):
    if not isinstance(texts, list):
        return []
    return [TextBlob(review).sentiment.polarity for review in texts]


In [42]:
sample_df["tb_scores"] = sample_df["clean_texts"].apply(get_textblob_sentiment)

In [43]:
def aggregate_textblob_sentiment(tb_scores):
    if not isinstance(tb_scores, list) or len(tb_scores) == 0:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])

    tb_avg = sum(tb_scores) / len(tb_scores)
    pos = sum(1 for s in tb_scores if s >= 0.05) / len(tb_scores)
    neg = sum(1 for s in tb_scores if s <= -0.05) / len(tb_scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([tb_avg, pos, neg, label], 
                     index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])


In [44]:
sample_df[["tb_avg", "tb_pos", "tb_neg", "tb_label"]] = (
    sample_df["tb_scores"].apply(aggregate_textblob_sentiment)
)

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

#### BERT

In [45]:
pip install transformers torch


Note: you may need to restart the kernel to use updated packages.


In [46]:
from transformers import pipeline

# Load BERT sentiment analysis pipeline
bert_classifier = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
def get_bert_sentiment(cleaned_reviews):
    if not isinstance(cleaned_reviews, list):
        return pd.Series([[], 0.0, 0.0, 0.0, "neutral"],
                         index=["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"])

    labels = []
    scores = []

    for review in cleaned_reviews:
        if isinstance(review, str) and review.strip():
            pred = bert_classifier(review[:512])[0]
            label = pred["label"].lower()
            score = pred["score"]

            labels.append(label)
            if label == "positive":
                scores.append(score)
            elif label == "negative":
                scores.append(-score)

    total = len(labels)
    pos = labels.count("positive")
    neg = labels.count("negative")
    pct_pos = pos / total if total > 0 else 0.0
    pct_neg = neg / total if total > 0 else 0.0
    avg_score = round(sum(scores) / len(scores), 4) if scores else 0.0

    if pos > neg:
        overall = "positive"
    elif neg > pos:
        overall = "negative"
    else:
        overall = "neutral"

    return pd.Series([labels, avg_score, pct_pos, pct_neg, overall],
                     index=["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"])


In [48]:
sample_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = sample_df["clean_texts"].apply(get_bert_sentiment)


In [49]:
sample_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,...,tb_scores,tb_avg,tb_pos,tb_neg,tb_label,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,5,[Fair prices and great service! Food was tasty...,The Corner Bar and Eatery,"163 Queens Road, Panmure, Auckland 1072, New Z...",-36.901067,174.856835,144,restaurant,[fair prices and great service food was tasty ...,...,"[0.5, 0.4375000000000001, 0.0, 0.4, 0.06666666...",0.280833,0.8,0.0,positive,"[positive, positive, positive, positive, negat...",0.599,0.8,0.2,positive


In [50]:
# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

#### MODEL COMPARISON

In [51]:
# Load the file
comp_df = pd.read_excel("manual labels for reviews.xlsx")

In [52]:
man_agg = []
for i in range(len(comp_df)):
    labels = comp_df.loc[i, "manual_labels"].split(",")
    labels = [l.strip().lower() for l in labels]  # normalize case just in case
    counts = pd.Series(labels).value_counts()

    if (counts == counts.max()).sum() > 1:
        agg = "neutral"
    else:
        agg = counts.idxmax()

    man_agg.append(agg)

comp_df["man_agg"] = man_agg


In [53]:
comp_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,keywords,sentiment_label,tb_label,bert_label,manual_labels,man_agg
0,ChIJ0e989wk7DW0RnfvTQBhLL4I,2,['Chicken pies tonight.\nStaff seemed a bit gr...,Kats Kitchen,"75 Oaktree Avenue, Browns Bay, Auckland 0630, ...",-36.722304,174.731495,2,restaurant,['chicken pies tonight staff seemed a bit grum...,[],neutral,neutral,neutral,"negative, positive",neutral


In [54]:
from sklearn.metrics import accuracy_score

# Compare manual labels vs each model
vader_acc = accuracy_score(comp_df["man_agg"], comp_df["sentiment_label"])
textblob_acc = accuracy_score(comp_df["man_agg"], comp_df["tb_label"])
bert_acc = accuracy_score(comp_df["man_agg"], comp_df["bert_label"])

# Print results
print("Accuracy vs Manual Labels:")
print(f"VADER Accuracy     : {vader_acc:.2f}")
print(f"TextBlob Accuracy  : {textblob_acc:.2f}")
print(f"BERT Accuracy      : {bert_acc:.2f}")


Accuracy vs Manual Labels:
VADER Accuracy     : 0.87
TextBlob Accuracy  : 0.97
BERT Accuracy      : 0.97


In [55]:
comp_df['man_agg'].describe()

count           30
unique           3
top       positive
freq            24
Name: man_agg, dtype: object

**Data is imbalanced. Positive counts are 24/30, so accuracy alone is misleading. Therefore classification report can provide deeper insight on model selection**

In [56]:
from sklearn.metrics import classification_report

print("VADER:")
print(classification_report(comp_df["man_agg"], comp_df["sentiment_label"]))

print("TextBlob:")
print(classification_report(comp_df["man_agg"], comp_df["tb_label"]))

print("BERT:")
print(classification_report(comp_df["man_agg"], comp_df["bert_label"]))


VADER:
              precision    recall  f1-score   support

    negative       1.00      0.20      0.33         5
     neutral       0.50      1.00      0.67         1
    positive       0.89      1.00      0.94        24

    accuracy                           0.87        30
   macro avg       0.80      0.73      0.65        30
weighted avg       0.89      0.87      0.83        30

TextBlob:
              precision    recall  f1-score   support

    negative       1.00      0.80      0.89         5
     neutral       1.00      1.00      1.00         1
    positive       0.96      1.00      0.98        24

    accuracy                           0.97        30
   macro avg       0.99      0.93      0.96        30
weighted avg       0.97      0.97      0.97        30

BERT:
              precision    recall  f1-score   support

    negative       0.83      1.00      0.91         5
     neutral       1.00      1.00      1.00         1
    positive       1.00      0.96      0.98        2

#### Classification Reports

**VADER**:
- **Strengths**: High accuracy on positive sentiment (Precision = 0.89, Recall = 1.00)
- **Weaknesses**: Very poor recall on negative reviews (Recall = 0.20), meaning it often misses them.
- **Neutral detection**: Correctly detected the only neutral sample.

**TextBlob**:
- **Strengths**: High precision and recall across all categories. Neutral and positive classes were perfectly predicted.
- **Weaknesses**: Slight underperformance on negative recall (0.80), but overall very strong.

**BERT**:
- **Strengths**: Most balanced performance, excellent at detecting both positive and negative reviews.
- **Weaknesses**: Missed just one positive review (Recall = 0.96).
- **Neutral detection**: Perfect.

---

#### Interpretation

- **Accuracy alone is misleading** due to class imbalance (80% positive).
- **VADER** struggles with subtle or implied negativity.
- **TextBlob** offers excellent performance with simple implementation.
- **BERT** is the most robust and balanced model — ideal for production-level sentiment tasks.

---

#### ✅ Final Model Selection: BERT

Based on the evaluation of sentiment models (VADER, TextBlob, and BERT) against manual labels:

- **BERT** provided the **most balanced and accurate** performance across all sentiment classes.
- It achieved:
  - **100% recall for negative and neutral** sentiments
  - **96% recall for positive** sentiments
  - **Overall accuracy of 97%**, matching TextBlob but with **stronger performance on the negative class**.

---

In [57]:
rest_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = rest_df["clean_texts"].apply(get_bert_sentiment)

# Save to Excel
rest_df.to_excel("restaurants_sentiment.xlsx", index=False)

In [58]:
rest_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
14,ChIJ-acdCmJJDW0R6nfpTBwEDSE,5,[Fair prices and great service! Food was tasty...,The Corner Bar and Eatery,"163 Queens Road, Panmure, Auckland 1072, New Z...",-36.901067,174.856835,144,restaurant,[fair prices and great service food was tasty ...,"[[fair, prices, great, service, food, tasty, g...","[amazing, clean, friendly, great, reasonable]","[positive, positive, positive, positive, negat...",0.599,0.8,0.2,positive


In [59]:
park_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = park_df["clean_texts"].apply(get_bert_sentiment)
park_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
0,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63,park,[my favourite place to take my little one for ...,"[[favourite, place, take, little, one, quick, ...","[maintained, tidy]","[positive, negative, positive, positive, posit...",0.5992,0.8,0.2,positive


In [60]:
# Save to Excel
park_df.to_excel("parks_sentiment.xlsx", index=False)

In [61]:
mall_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = mall_df["clean_texts"].apply(get_bert_sentiment)
mall_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
18,ChIJ0-4gpPJGDW0RUY3nD0-Pn1k,5,[This is a very nice center. There are not too...,Tulja Centre,"190 Stoddard Road, Wesley, Auckland 1041, New ...",-36.90068,174.721294,44,shopping mall,[this is a very nice center there are not too ...,"[[nice, center, many, businesses, couple, time...","[big, small]","[positive, positive, negative, positive, negat...",0.1993,0.6,0.4,positive


In [62]:
# Save to Excel
mall_df.to_excel("malls_sentiment.xlsx", index=False)

In [63]:
tour_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = tour_df["clean_texts"].apply(get_bert_sentiment)
tour_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
1,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87,tourist attraction,[great place for families and dog walkers benc...,"[[great, place, families, dog, walkers, benche...",[crowded],"[positive, positive, positive, positive, posit...",0.9989,1.0,0.0,positive


In [64]:
# Save to Excel
tour_df.to_excel("tourist_sentiment.xlsx", index=False)

In [65]:
# Concatenate all category dataframe
final_sent = pd.concat([rest_df, mall_df, tour_df, park_df], ignore_index=True)
final_sent.to_excel("sentiment_merged.xlsx", index=False)

### Aspect-Based Sentiment Analysis (ABSA)

In [66]:
# Install required libraries
!pip install transformers pandas openpyxl hf_xet --quiet 


In [67]:
from transformers import pipeline

# Use a small and public model (multilingual, fast)
sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The tokenizer you are loading from 'nlptown/bert-base-multilingual-uncased-sentiment' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [68]:
import ast

# Load your file
rest_aspect = pd.read_excel("restaurants_sentiment.xlsx")
# Convert stringified list to actual list
rest_aspect["clean_texts"] = rest_aspect["clean_texts"].apply(ast.literal_eval)

In [69]:
# Define mapping from model labels to numeric values
label_map = {
    "1 star": 1,
    "2 stars": 2,
    "3 stars": 3,
    "4 stars": 4,
    "5 stars": 5
}

# Function to score list of reviews
def score_reviews(review_list):
    results = sentiment_model(review_list)
    scores = [label_map[r['label']] for r in results]
    return {
        "avg_sentiment": sum(scores)/len(scores),
        "max_sentiment": max(scores),
        "min_sentiment": min(scores),
        "positive_pct": sum(1 for s in scores if s >= 4) / len(scores),
        "review_count": len(scores),
        "individual_scores": scores
    }
